In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('heart.csv')
print(f"Dataset loaded: {df.shape}")  # (918, 12)

Dataset loaded: (918, 12)


In [2]:
# 1 baris dengan RestingBP=0 (tidak mungkin secara klinis)
bp_median = df[df['RestingBP'] != 0]['RestingBP'].median()
df['RestingBP'] = df['RestingBP'].replace(0, bp_median)

assert (df['RestingBP'] == 0).sum() == 0, "Masih ada RestingBP=0!"
print(f"RestingBP fix: ✅  median={bp_median}")

RestingBP fix: ✅  median=130.0


In [3]:
# Tambahkan flag SEBELUM imputasi — ini menyimpan informasi MNAR
df['Cholesterol_missing'] = (df['Cholesterol'] == 0).astype(int)

chol_median = df[df['Cholesterol'] != 0]['Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, chol_median)

assert (df['Cholesterol'] == 0).sum() == 0, "Masih ada Cholesterol=0!"
print(f"Cholesterol_missing flag: {df['Cholesterol_missing'].sum()} baris ditandai")
print(f"Cholesterol imputasi: ✅  median={chol_median}")

Cholesterol_missing flag: 172 baris ditandai
Cholesterol imputasi: ✅  median=237.0


In [4]:
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

print(f"X shape sebelum OHE: {X.shape}")  # (918, 12) — termasuk Cholesterol_missing

X shape sebelum OHE: (918, 12)


In [5]:
X = pd.get_dummies(
    X,
    columns=['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'],
    drop_first=True
)

print(f"X shape setelah OHE: {X.shape}")  # (918, 16)
print("Kolom akhir:")
for i, col in enumerate(X.columns):
    print(f"  {i+1:2d}. {col}")

X shape setelah OHE: (918, 16)
Kolom akhir:
   1. Age
   2. RestingBP
   3. Cholesterol
   4. FastingBS
   5. MaxHR
   6. Oldpeak
   7. Cholesterol_missing
   8. Sex_M
   9. ChestPainType_ATA
  10. ChestPainType_NAP
  11. ChestPainType_TA
  12. RestingECG_Normal
  13. RestingECG_ST
  14. ExerciseAngina_Y
  15. ST_Slope_Flat
  16. ST_Slope_Up


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # jaga proporsi class di kedua split
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.value_counts().to_dict()}")
print(f"y_test  : {y_test.value_counts().to_dict()}")
# Class imbalance ratio 0.807 — tidak perlu SMOTE

X_train : (734, 16)
X_test  : (184, 16)
y_train : {1: 406, 0: 328}
y_test  : {1: 102, 0: 82}


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)         # transform ONLY — jangan fit!

assert hasattr(scaler, 'mean_'),  "Scaler belum di-fit!"
assert scaler.n_features_in_ == 16, f"Harusnya 16 fitur, dapat {scaler.n_features_in_}"
print("Scaler fitted: ✅")

Scaler fitted: ✅


In [8]:
feature_names = list(X_train.columns)

X_train_final = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_final  = pd.DataFrame(X_test_scaled,  columns=feature_names)
y_train_final = pd.Series(y_train.values, name='HeartDisease')
y_test_final  = pd.Series(y_test.values,  name='HeartDisease')

In [9]:
X_train_final.to_csv('X_train.csv', index=False)
X_test_final.to_csv('X_test.csv',   index=False)
y_train_final.to_csv('y_train.csv', index=False)
y_test_final.to_csv('y_test.csv',   index=False)

joblib.dump(scaler, 'scaler.joblib')

with open('feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

print("✅ File tersimpan di direktori aktif:")
print("   X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("   scaler.joblib, feature_names.txt")

✅ File tersimpan di direktori aktif:
   X_train.csv, X_test.csv, y_train.csv, y_test.csv
   scaler.joblib, feature_names.txt


In [ ]:
s = joblib.load('scaler.joblib')
assert hasattr(s, 'mean_'),         "Scaler belum di-fit!"
assert s.n_features_in_ == 16,     "Feature count salah!"

Xtr = pd.read_csv('X_train.csv')
Xte = pd.read_csv('X_test.csv')
assert Xtr.shape[1] == 16,                         "Harusnya 16 kolom!"
assert list(Xtr.columns) == list(Xte.columns),     "Kolom tidak match!"
assert 'Cholesterol_missing' in Xtr.columns,       "MNAR flag tidak ada!"

print("✅ Semua verifikasi passed")
print(f"   X_train : {Xtr.shape} | X_test : {Xte.shape}")
print(f"   y_train balance : {pd.read_csv('y_train.csv').squeeze().value_counts().to_dict()}")

✅ Semua verifikasi passed
   X_train : (734, 16) | X_test : (184, 16)
   y_train balance : {1: 406, 0: 328}
